# From-Scratch Hidden Markov Models for Object Recognition in Images

_Generated: 2025-10-22T16:08:00.136770Z_

**Goal.** Implement HMMs from first principles (no HMM libraries) and use them for image object recognition.

**High-level plan**
1. Convert each image into a **sequence** by scanning columns left→right. Each time step is the column's pixel vector.
2. Train a **class-conditional HMM** per object class (here: handwritten digits 0–9).
3. Use **Baum–Welch (EM)** with **diagonal Gaussian emissions** to estimate parameters for each class.
4. Classify a test image by the **maximum log-likelihood** under the class HMMs.

**Why this works.** Many objects exhibit sequential structure along one axis (e.g., strokes evolving horizontally). An HMM captures this through latent "parts" (states) and a Markovian progression.

**From-first-principles constraint.** We use `numpy` for all HMM internals (forward/backward, EM, Viterbi). `scikit-learn` is only used to **fetch data** and optionally to **verify metrics**.

## 0. Runtime & Environment Check

This notebook is CPU-friendly; GPU is not required.

In [ ]:
import sys, platform, subprocess

def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)
print("\nNVIDIA SMI (if present):")
_sh("nvidia-smi || true")

## 1. Minimal Dependencies

We rely on `numpy`, `matplotlib`, and `scikit-learn` (datasets & metrics only).

In [ ]:
!pip -q install --upgrade pip
!pip -q install numpy matplotlib scikit-learn

## 2. Imports & Reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 123
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
print("Seeded.")

## 3. Data: Images → Sequences (column scan)

We use the `digits` dataset (8×8 grayscale, values 0–16). Each image becomes a sequence of length **T=8**; each observation is a **D=8** vector corresponding to one column's intensities.

We normalize pixel values to [0,1].

In [ ]:
digits = load_digits()
X_images = digits.images.astype(np.float64)  # shape (N, 8, 8)
y_labels = digits.target.astype(int)

X_images /= 16.0  # normalize to [0,1]

def image_to_sequence(img):
    # img: (8,8). Sequence is list/array of length 8; each obs is (8,)
    # time t corresponds to column t
    return img.T  # (8,8): columns become rows (T,D)

sequences = [image_to_sequence(img) for img in X_images]  # list of (T=8, D=8)

print("Total images:", len(sequences))
print("Example sequence shape:", sequences[0].shape)

## 4. Train/Test Split (Stratified by Class)

In [ ]:
def stratified_split_indices(y, test_size=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    classes = np.unique(y)
    train_idx, test_idx = [], []
    for c in classes:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        n_test = int(np.floor(test_size * len(idx_c)))
        test_idx.extend(idx_c[:n_test].tolist())
        train_idx.extend(idx_c[n_test:].tolist())
    return np.array(train_idx), np.array(test_idx)

train_idx, test_idx = stratified_split_indices(y_labels, test_size=0.2, seed=SEED)
X_train = [sequences[i] for i in train_idx]
y_train = y_labels[train_idx]
X_test  = [sequences[i] for i in test_idx]
y_test  = y_labels[test_idx]

print("Train:", len(X_train), "Test:", len(X_test))

## 5. HMM Model (Diagonal-Gaussian Emissions)

**Parameters per class**
- Initial state distribution: $\pi \in \mathbb{R}^K$ (simplex)
- Transition matrix: $A \in \mathbb{R}^{K\times K}$ (row-stochastic)
- Emissions: for each state k, a diagonal-Gaussian $\mathcal{N}(\mu_k, \mathrm{diag}(\sigma_k^2))$ over $\mathbb{R}^D$

**Learning**: Baum–Welch (EM)
1. E-step: compute scaled forward/backward to get $\gamma_t(k)=p(z_t=k|x_{1:T})$ and $\xi_t(i,j)$.
2. M-step: update $\pi, A, (\mu_k, \sigma_k^2)$ using expected sufficient statistics.

**Numerics**: we use scaling factors in forward/backward to avoid underflow. Log-likelihood is the sum of log-scales.

### 5.1 Implementation (from scratch)

In [ ]:
import numpy as np

def normalize_rows(M, eps=1e-12):
    M = np.asarray(M, dtype=np.float64)
    row_sums = M.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums < eps, 1.0, row_sums)
    return M / row_sums

def init_hmm_params(K, D, rng):
    # Random row-stochastic A, uniform pi, random means, unit variances
    A = rng.random((K, K)) + 1e-2
    A = normalize_rows(A)
    pi = np.ones(K) / K
    means = rng.normal(0.0, 0.5, size=(K, D))
    vars_ = np.ones((K, D)) * 0.25  # reasonable initial variance
    return pi, A, means, vars_

def log_diag_gaussian(x, mean, var, eps=1e-6):
    # x: (D,), mean: (D,), var: (D,)
    var = np.maximum(var, eps)
    D = x.shape[-1]
    logdet = np.sum(np.log(var))
    quad = np.sum((x - mean)**2 / var)
    return -0.5 * (logdet + quad + D*np.log(2*np.pi))

def emission_loglik_sequence(seq, means, vars_):
    # seq: (T,D); means: (K,D); vars_: (K,D)
    T, D = seq.shape
    K = means.shape[0]
    B = np.empty((T, K), dtype=np.float64)
    for t in range(T):
        for k in range(K):
            B[t, k] = log_diag_gaussian(seq[t], means[k], vars_[k])
    return B  # log-likelihoods per state

def forward_backward_scaled(logB, pi, A, eps=1e-12):
    # logB: (T,K) log emission likelihoods; pi: (K,), A: (K,K)
    T, K = logB.shape
    # Precompute emissions in linear domain with care via scaling
    # We'll keep alpha, beta in linear domain but scale by c_t.
    B = np.exp(logB - logB.max(axis=1, keepdims=True))  # stabilize exponent
    # Forward
    alpha = np.zeros((T, K), dtype=np.float64)
    c = np.zeros(T, dtype=np.float64)
    alpha[0] = pi * B[0]
    c[0] = alpha[0].sum() + eps
    alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[t]
        c[t] = alpha[t].sum() + eps
        alpha[t] /= c[t]
    # Backward
    beta = np.zeros((T, K), dtype=np.float64)
    beta[-1] = 1.0 / c[-1]
    for t in range(T-2, -1, -1):
        beta[t] = (A @ (B[t+1] * beta[t+1]))
        beta[t] /= c[t]
    # Gammas and Xis
    gamma = alpha * beta  # automatically normalized
    gamma /= gamma.sum(axis=1, keepdims=True)
    xi = np.zeros((T-1, K, K), dtype=np.float64)
    for t in range(T-1):
        # unnormalized xi
        z = (alpha[t][:,None] * A) * (B[t+1][None,:] * beta[t+1][None,:])
        s = z.sum() + eps
        xi[t] = z / s
    # Log-likelihood: sum log c_t + corrections from exp-stabilization (cancel in scaling)
    loglik = np.sum(np.log(c + eps)) + np.sum(logB.max(axis=1))
    return alpha, beta, gamma, xi, loglik

def baum_welch_diagonal_gaussian(seqs, K, iters=20, tol=1e-4, min_var=1e-4, rng=np.random.default_rng(0), verbose=True):
    # seqs: list of arrays, each (T,D)
    D = seqs[0].shape[1]
    pi, A, means, vars_ = init_hmm_params(K, D, rng)
    prev_ll = -np.inf
    ll_history = []
    for it in range(1, iters+1):
        # E-step accumulators
        pi_acc = np.zeros(K)
        A_acc = np.zeros((K, K))
        mean_num = np.zeros((K, D))
        var_num = np.zeros((K, D))
        gamma_den = np.zeros(K)
        total_ll = 0.0

        for seq in seqs:
            logB = emission_loglik_sequence(seq, means, vars_)
            alpha, beta, gamma, xi, ll = forward_backward_scaled(logB, pi, A)
            total_ll += ll
            # Accumulate
            pi_acc += gamma[0]
            A_acc += xi.sum(axis=0)
            for k in range(K):
                gk = gamma[:,k][:,None]  # (T,1)
                mean_num[k] += (gk * seq).sum(axis=0)
                var_num[k]  += (gk * (seq - means[k])**2).sum(axis=0)  # will re-compute after mean update; placeholder
                gamma_den[k] += gamma[:,k].sum()

        # M-step
        pi = pi_acc / pi_acc.sum()
        A = normalize_rows(A_acc + 1e-8)  # add tiny smoothing
        # Update means
        new_means = np.zeros_like(means)
        for k in range(K):
            if gamma_den[k] < 1e-8:
                new_means[k] = means[k]
            else:
                new_means[k] = mean_num[k] / gamma_den[k]
        means = new_means
        # Update variances
        var_num = np.zeros_like(vars_)
        for seq in seqs:
            logB = emission_loglik_sequence(seq, means, vars_)  # means updated
            alpha, beta, gamma, xi, ll_dummy = forward_backward_scaled(logB, pi, A)
            for k in range(K):
                gk = gamma[:,k][:,None]
                var_num[k] += (gk * (seq - means[k])**2).sum(axis=0)
        new_vars = np.zeros_like(vars_)
        for k in range(K):
            if gamma_den[k] < 1e-8:
                new_vars[k] = vars_[k]
            else:
                new_vars[k] = var_num[k] / gamma_den[k]
            new_vars[k] = np.maximum(new_vars[k], min_var)
        vars_ = new_vars

        ll_history.append(total_ll)
        if verbose:
            print(f"[EM] iter {it:02d}  loglik={total_ll:.3f}")
        if np.isfinite(prev_ll) and (total_ll - prev_ll) < tol:
            break
        prev_ll = total_ll
    return {"pi": pi, "A": A, "means": means, "vars": vars_, "ll_history": ll_history}

def sequence_loglik(seq, model):
    logB = emission_loglik_sequence(seq, model["means"], model["vars"])
    _, _, _, _, ll = forward_backward_scaled(logB, model["pi"], model["A"])
    return ll

def viterbi_path(seq, model):
    # For illustration (most useful for qualitative state paths)
    pi, A, means, vars_ = model["pi"], model["A"], model["means"], model["vars"]
    T = seq.shape[0]; K = means.shape[0]
    logB = emission_loglik_sequence(seq, means, vars_)
    log_pi = np.log(pi + 1e-12)
    log_A = np.log(A + 1e-12)
    delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)

    delta[0] = log_pi + logB[0]
    for t in range(1, T):
        for k in range(K):
            prev = delta[t-1] + log_A[:,k]
            psi[t, k] = int(np.argmax(prev))
            delta[t, k] = prev[psi[t,k]] + logB[t, k]
    path = np.zeros(T, dtype=int)
    path[-1] = int(np.argmax(delta[-1]))
    for t in range(T-2, -1, -1):
        path[t] = psi[t+1, path[t+1]]
    return path, delta.max(axis=1).sum()

## 6. Training Class-Conditional HMMs

We train an HMM per digit class with K latent states (e.g., K=4). To keep runtime reasonable, you may subsample training sequences per class with `max_per_class`.

In [ ]:
K = 4               # number of latent states per class
EM_ITERS = 20       # EM iterations
MAX_PER_CLASS = None  # e.g., 120 to cap per class; None for all

# group sequences by class
class_seqs = {c: [] for c in range(10)}
for seq, lab in zip(X_train, y_train):
    class_seqs[int(lab)].append(seq)

if MAX_PER_CLASS is not None:
    for c in class_seqs:
        rng.shuffle(class_seqs[c])
        class_seqs[c] = class_seqs[c][:MAX_PER_CLASS]

models = {}
for c in range(10):
    print(f"Training HMM for class {c} with {len(class_seqs[c])} sequences...")
    models[c] = baum_welch_diagonal_gaussian(
        class_seqs[c], K=K, iters=EM_ITERS, tol=1e-3, rng=rng, verbose=True
    )
print("Done.")

### 6.1 EM Convergence Diagnostics

In [ ]:
fig = plt.figure(figsize=(6,4))
for c in range(10):
    ll = models[c]["ll_history"]
    plt.plot(ll, label=str(c))
plt.xlabel("EM iteration")
plt.ylabel("Log-likelihood (sum over class sequences)")
plt.title("EM Convergence per Class HMM")
plt.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()

## 7. Classification by Maximum Likelihood

Predict the class label by computing the sequence log-likelihood under each class HMM and choosing the argmax.

In [ ]:
def predict(seq, models):
    scores = np.array([sequence_loglik(seq, models[c]) for c in range(10)])
    return int(np.argmax(scores)), scores

y_pred = []
all_scores = []
for seq in X_test:
    label, scores = predict(seq, models)
    y_pred.append(label)
    all_scores.append(scores)

y_pred = np.array(y_pred)
all_scores = np.vstack(all_scores)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
fig = plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion Matrix (HMM columns-as-sequence)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.tight_layout()
plt.show()

### 7.1 Qualitative: Viterbi State Paths

For a few test images, visualize the learned state path across columns (time).

In [ ]:
def show_image_and_path(seq, true_label, pred_label, path):
    # seq is (T=8, D=8); show original image (transpose back) and state path
    img = seq.T  # back to (8,8)
    fig = plt.figure(figsize=(4,3))
    plt.subplot(2,1,1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.axis('off')
    plt.title(f"True {true_label} / Pred {pred_label}")
    plt.subplot(2,1,2)
    plt.plot(path, marker='o')
    plt.ylim(-0.5, max(path)+0.5)
    plt.yticks(range(max(path)+1))
    plt.xlabel("Time (column index)"); plt.ylabel("State")
    plt.tight_layout()
    plt.show()

for i in range(5):
    seq = X_test[i]
    tl = y_test[i]
    pl = y_pred[i]
    path, _ = viterbi_path(seq, models[pl])
    show_image_and_path(seq, tl, pl, path)

## 8. Save Artifacts & Download

We persist trained parameters for each class and evaluation metrics. Use the cell below in Colab to download.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

# Save models
for c in range(10):
    np.savez(f"artifacts/hmm_class_{c}.npz",
             pi=models[c]["pi"], A=models[c]["A"],
             means=models[c]["means"], vars=models[c]["vars"],
             ll_history=np.array(models[c]["ll_history"], dtype=np.float64))

# Save metrics
metrics = {
    "accuracy": float(acc),
    "confusion_matrix": cm.tolist()
}
with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved files:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 9. Appendix: Extensions & Variations

- **Richer emissions**: full-covariance Gaussians; mixtures per state (GMM-HMM).
- **2D sequences**: combine row and column scans or use snake-like scans; average log-likelihoods.
- **Discretization**: vector-quantize columns into codewords; use discrete-emission HMM.
- **Segmentation**: use Viterbi paths to segment images into latent parts; align across classes.
- **Initialization**: from scratch k-means on columns to initialize means/variances per state.
- **Regularization**: Dirichlet priors on $\pi$ and $A$, variance floors, early stopping.
- **Larger images**: downsample to manageable column vectors or extract patch-strips.